# NovaFin Group capstone — run the whole project

**What this does:** runs every module in order (1 → 9), each through its own runner in its own folder, then rebuilds the three final deliverables in `final_deliverables/` (final report and code guide as PDF, Board presentation as PowerPoint).

**How to use:**
- **Run all** to reproduce the entire project (about **2 hours**: Module 5 ≈ 50 min, Module 1 ≈ 20 min, Module 2 ≈ 9 min, Modules 3 and 4 ≈ 13 min each, Modules 6–9 ≈ 7 min together), or run any single cell to rerun one module.
- **Order matters:** later modules read files earlier ones save (e.g. Module 6 uses Module 5's predictions; Module 9 uses every module's `module9_inputs.csv`). Running one module on its own uses the other modules' saved outputs.
- Every module also runs on its own: open `module_0N_*/run_module_N.ipynb`.
- **Nothing needs to be run to see the results:** the tables and charts are saved inside every `module_0N_*/run_module_N.ipynb` and task notebook, and in the report PDFs. This notebook only starts the modules, so it is saved without outputs.
- Without opening it: `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 run_novafin.ipynb`

**Reproducible:** seed 42 everywhere; each module's runner reproduces its standalone notebooks exactly (checked file by file).

## Where to see the results

Every file below is already saved in the project folder; running the project rewrites it in the same place (in Google Colab: in the project folder in your Google Drive).

| Read in this order | File | What it shows |
| --- | --- | --- |
| 1. Final report | `final_deliverables/NovaFin_Final_Report.pdf` | The whole project in words, tables and charts: how the ₹1,000 crore is used, why, and how sure we are |
| 2. Board presentation | `final_deliverables/NovaFin_Board_Presentation.pptx` | The same conclusions in 16 slides, with speaker notes |
| 3. Module workbooks | `module_0N_*/outputs/FINAL_MN_module_output.xlsx` | Each module's results and decisions row by row; start with its "1 Read me" tab |
| 4. Module reports | `module_0N_*/reports/module_N_report.pdf` | Each module explained in full, with its charts |
| 5. Module runner notebooks | `module_0N_*/run_module_N.ipynb` | The code with every table and chart it produced |

- **N is the module number** (1 to 9): for Module 3, `module_03_fraud_detection/outputs/FINAL_M3_module_output.xlsx`.
- **While this notebook runs:** each module's tables and charts appear under its cell, and a message at the end of the cell names the module's workbook and report.
- **When it finishes:** the last cell lists every final file, what it shows and when it was saved.
- **What every other file is for:** `HOW_TO_RUN.md` (section 0 lists the final files; section 4 lists all files).

## Setup
Finds the project folder (any folder name, locally or in Google Colab) and defines `run_module(n)`, which moves into that module's folder, runs its runner, moves back, and says which files hold the results.

In [ ]:
import os
import sys
import time
from pathlib import Path

# ---- Find the project folder: any folder name works, on your computer or in Google Colab ----
# The project folder is the one holding run_novafin.ipynb and data/.
IN_COLAB = "google.colab" in sys.modules
is_project = lambda p: (p / "run_novafin.ipynb").exists() and (p / "data").is_dir()
places = [p for p in [Path.cwd(), *Path.cwd().parents] if is_project(p)]
if IN_COLAB and not places:                                 # Colab starts outside the project: look in /content, then in Google Drive
    for base in [Path("/content"), Path("/content/drive/MyDrive")]:
        if "drive" in base.parts:
            from google.colab import drive
            drive.mount("/content/drive")                   # asks for access to your Google Drive
        for depth in ["", "*/", "*/*/", "*/*/*/"]:
            places += [f.parent for f in sorted(base.glob(depth + "run_novafin.ipynb")) if is_project(f.parent)]
        if places:
            break
if not places:
    raise FileNotFoundError("Project folder not found: unzip the whole project and keep run_novafin.ipynb, data/ and the module folders together (see HOW_TO_RUN.md).")
PROJECT_ROOT = places[0]
os.chdir(PROJECT_ROOT)
MODULE_FOLDERS = {1: "module_01_corporate_finance", 2: "module_02_credit_risk", 3: "module_03_fraud_detection", 4: "module_04_customer_analytics", 5: "module_05_equity_markets", 6: "module_06_portfolio_optimization", 7: "module_07_hft", 8: "module_08_derivatives", 9: "module_09_capital_allocation"}
MODULE_SHOWS = {1: "Which initiatives to fund, review or reject, and why", 2: "Default risk of every loan, and which to approve, review or reject", 3: "Fraud risk of every transaction, and which to investigate", 4: "Churn risk and value of every customer, and which ones to contact", 5: "Predicted returns, stock rankings and test portfolios", 6: "Portfolio weights and the portfolio's risk", 7: "Price-direction predictions and simulated trade decisions", 8: "Option prices (Black-Scholes and ML) and hedging", 9: "How the ₹1,000 crore is allocated, the reserve plan, and the stress tests"}   # what each module's workbook holds
MODULE_MINUTES = {}


def result_files(n: int) -> dict:
    """Module n's workbook and report, as paths inside the project folder."""
    return {"Workbook": f"{MODULE_FOLDERS[n]}/outputs/FINAL_M{n}_module_output.xlsx", "Report": f"{MODULE_FOLDERS[n]}/reports/module_{n}_report.pdf"}


def run_module(n: int) -> None:
    """Run module n's runner notebook inside its own folder (its paths are relative to it), then return to the project folder."""
    folder = PROJECT_ROOT / MODULE_FOLDERS[n]
    start = time.time()
    os.chdir(folder)
    try:
        get_ipython().run_line_magic("run", f"run_module_{n}.ipynb")
    finally:
        os.chdir(PROJECT_ROOT)
    MODULE_MINUTES[n] = round((time.time() - start) / 60, 1)
    print(f"Module {n} finished in {MODULE_MINUTES[n]} minutes. Where to see the results:")
    print("  Tables and charts: above, in this cell's output")
    for kind, path in result_files(n).items():
        print(f"  {kind}: {path}")


print("Project folder found")

## Install the libraries
Installs the library versions the project was tested with (`requirements.txt`), so no separate `pip install` is needed. Takes a minute the first time and a few seconds after that. In Google Colab the session may restart once after this step: choose **Run all** again.

In [ ]:
%pip install -q -r requirements.txt
# Google Colab loads some libraries before this cell runs. If a version changed, the session restarts once: then choose Run all again.
import importlib.metadata
changed = [m for m in ["numpy", "pandas", "scipy", "matplotlib"] if m in sys.modules and sys.modules[m].__version__ != importlib.metadata.version(m)]
if IN_COLAB and changed:
    print("Libraries updated:", ", ".join(changed), "- the session now restarts. When it is back, choose Run all again.")
    time.sleep(3)
    os.kill(os.getpid(), 9)

## Module 1 — Corporate finance: strategic initiatives
Runs `module_01_corporate_finance/run_module_1.ipynb` (about 19 minutes).

In [ ]:
run_module(1)

## Module 2 — Credit risk
Runs `module_02_credit_risk/run_module_2.ipynb` (about 9 minutes).

In [ ]:
run_module(2)

## Module 3 — Fraud detection
Runs `module_03_fraud_detection/run_module_3.ipynb` (about 13 minutes).

In [ ]:
run_module(3)

## Module 4 — Customer analytics
Runs `module_04_customer_analytics/run_module_4.ipynb` (about 13 minutes).

In [ ]:
run_module(4)

## Module 5 — Equity markets
Runs `module_05_equity_markets/run_module_5.ipynb` (about 50 minutes).

In [ ]:
run_module(5)

## Module 6 — Portfolio optimisation and risk
Runs `module_06_portfolio_optimization/run_module_6.ipynb` (under a minute).

In [ ]:
run_module(6)

## Module 7 — High-frequency trading
Runs `module_07_hft/run_module_7.ipynb` (about 4 minutes).

In [ ]:
run_module(7)

## Module 8 — Derivatives
Runs `module_08_derivatives/run_module_8.ipynb` (about 3 minutes).

In [ ]:
run_module(8)

## Module 9 — Integrated capital allocation of ₹1,000 crore
Runs `module_09_capital_allocation/run_module_9.ipynb` (under a minute).

In [ ]:
run_module(9)

## The final deliverables
Rebuilds `final_deliverables/NovaFin_Final_Report.pdf`, `NovaFin_Code_Guide.pdf` and `NovaFin_Board_Presentation.pptx` and the PDF of every module and task report, from the modules' saved outputs, so they always match the notebooks.

In [ ]:
os.chdir(PROJECT_ROOT)
get_ipython().run_line_magic("run", "scripts/build_final_deliverables.py")
get_ipython().run_line_magic("run", "scripts/build_board_deck.py")

## Summary: the final files, in reading order
What ran, how long it took, and where to see the results.

In [ ]:
import pandas as pd
from datetime import datetime
from IPython.display import HTML, display

print("Minutes per module:", MODULE_MINUTES)
# Reading order: the final report, the Board deck and the code guide, then each module's workbook and report
rows = [("Whole project", "final_deliverables/NovaFin_Final_Report.pdf", "Open first: the whole project, the plan for the ₹1,000 crore and the reasons"),
        ("Whole project", "final_deliverables/NovaFin_Board_Presentation.pptx", "The Board presentation, with speaker notes"),
        ("Whole project", "final_deliverables/NovaFin_Code_Guide.pdf", "How the code is organised and how to run it")]
for n in MODULE_FOLDERS:
    f = result_files(n)
    rows += [(f"Module {n}", f["Workbook"], MODULE_SHOWS[n]), (f"Module {n}", f["Report"], "The full explanation, with charts")]
saved = lambda f: datetime.fromtimestamp((PROJECT_ROOT / f).stat().st_mtime).strftime("%Y-%m-%d %H:%M") if (PROJECT_ROOT / f).exists() else "NOT FOUND"
finals = pd.DataFrame([(part, f, what, saved(f)) for part, f, what in rows], columns=["Part", "Final file (inside the project folder)", "What it shows", "Last saved"])
display(HTML(finals.to_html(index=False).replace("<td>", '<td style="text-align: left">').replace("<th>", '<th style="text-align: left">')))   # to_html shows full file names